# Data Chunking Pipeline — Huawei Ascend RAG

**Objetivo:** Conversão estruturada de manuais técnicos Huawei (PDF → Markdown) e laboratório comparativo de estratégias de *chunking* para sistemas de Retrieval-Augmented Generation (RAG).

**Seções do notebook:**
- **Seção A:** Configuração do ambiente, dependências e persistência
- **Seção B:** Extração de documentos PDF → Markdown com pós-processamento
- **Seção C:** Laboratório comparativo de estratégias de chunking

**Hardware alvo:** Ascend 310 AI Processor, Atlas 200 DK (Modelos 3000), DaVinci AI Cores  
**Software stack:** CANN/DDK, ATC/OMG, AscendCL, TE/TVM, AIPP

---
## 🔧 Seção A: Configuração do Ambiente e Dependências
*Setup do runtime Colab (GPU T4/V100), instalação de bibliotecas para parsing de layout, tokenização BPE e splitting de texto.*

In [26]:
# ============================================================================
# Cell A.1 — GPU Hardware Verification
# ============================================================================
import subprocess

def check_gpu() -> bool:
    """Check GPU availability and print hardware summary."""
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
             "--format=csv,noheader"],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0 and result.stdout.strip():
            print("GPU Detected:")
            print(result.stdout.strip())
            return True
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass
    print("No GPU detected — running on CPU (marker-pdf will be slower)")
    return False

HAS_GPU = check_gpu()

✅ GPU Detected:
NVIDIA A100-SXM4-80GB, 81920 MiB, 580.82.07


### 📦 Instalação de Dependências

Instalação das bibliotecas principais:
- **`marker-pdf`** — extração *layout-aware* de PDFs, preservando tabelas pipe de Markdown e hierarquia de cabeçalhos em documentos multi-coluna
- **`tiktoken`** — tokenizador BPE (Byte-Pair Encoding) para contagem precisa de tokens
- **`langchain-text-splitters`** — implementações de estratégias de *chunking* (recursivo, por cabeçalhos Markdown)
- **`tqdm`** — barras de progresso para loops de processamento

In [27]:
# ============================================================================
# Cell A.2 — Dependency Installation
# ============================================================================
%%capture install_output
!pip install -q "marker-pdf[full]" tiktoken langchain-text-splitters tqdm

In [28]:
# ============================================================================
# Cell A.2b — Verify Critical Imports
# ============================================================================
import importlib

print("Dependency verification:")
for pkg in ["marker", "tiktoken", "langchain_text_splitters", "tqdm"]:
    spec = importlib.util.find_spec(pkg)
    status = "OK" if spec else "MISSING"
    print(f"  {status}  {pkg}")

📋 Dependency verification:
  ✅  marker
  ✅  tiktoken
  ✅  langchain_text_splitters
  ✅  tqdm


In [29]:
# ============================================================================
# Cell A.3 — Core Imports
# ============================================================================
import os
import re
import json
import shutil
import zipfile
import textwrap
import warnings
from pathlib import Path
from typing import Optional
from collections import Counter, defaultdict

import tiktoken
from tqdm.auto import tqdm
from IPython.display import display, Markdown as IPyMarkdown, HTML

warnings.filterwarnings("ignore", category=UserWarning)
print("Core imports loaded")

✅ Core imports loaded


### 💾 Configuração de Persistência e Google Drive

O Google Drive é montado **antes** da inicialização de diretórios para que os PDFs fonte sejam lidos diretamente do Drive.
- `USE_GOOGLE_DRIVE = True` — monta o Drive e sincroniza artefatos automaticamente
- `USE_GOOGLE_DRIVE = False` — modo local com download via `.zip`

In [30]:
# ============================================================================
# Cell A.4 — Persistence / Google Drive Configuration
# ============================================================================
USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}
GDRIVE_DEST = "/content/drive/MyDrive/RAG_Atlas200DK/output"

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        Path(GDRIVE_DEST).mkdir(parents=True, exist_ok=True)
        print(f"Google Drive mounted -> {GDRIVE_DEST}")
    except ValueError:
        print("Google Drive already mounted (or mountpoint contains files) -> Skipping mount.")
        Path(GDRIVE_DEST).mkdir(parents=True, exist_ok=True)
    except ImportError:
        print("Not running in Colab — Drive mount skipped")
        USE_GOOGLE_DRIVE = False
else:
    print("Local mode — use zip download when ready")


def export_artifact(src_dir: Path, zip_name: str = "processed_markdown.zip"):
    """Pack src_dir into a zip and either download or copy to Drive."""
    zip_path = OUTPUT_DIR / zip_name
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fp in sorted(src_dir.rglob("*")):
            if fp.is_file():
                zf.write(fp, fp.relative_to(src_dir.parent))
    size_kb = zip_path.stat().st_size / 1024
    print(f"Created {zip_path} ({size_kb:.0f} KB)")

    if USE_GOOGLE_DRIVE:
        dest = Path(GDRIVE_DEST) / zip_name
        shutil.copy2(zip_path, dest)
        print(f"Copied to Google Drive: {dest}")
    else:
        try:
            from google.colab import files
            files.download(str(zip_path))
            print("Download triggered")
        except ImportError:
            print(f"Not in Colab — file available at: {zip_path.resolve()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted -> /content/drive/MyDrive/RAG_Atlas200DK/output


In [31]:
# ============================================================================
# Cell A.5 — Directory Structure Initialization
# ============================================================================
# When USE_GOOGLE_DRIVE is True, BASE_DIR points to Drive so PDFs are read
# directly from Drive and outputs are saved there.
if USE_GOOGLE_DRIVE:
    BASE_DIR = Path("/content/drive/MyDrive/RAG_Atlas200DK/")
else:
    BASE_DIR = Path(".")

RAW_PDF_DIR      = BASE_DIR / "data" / "raw" / "manuals"
PROCESSED_MD_DIR = BASE_DIR / "data" / "processed" / "markdown"
OUTPUT_DIR       = BASE_DIR / "data" / "output"

for d in [RAW_PDF_DIR, PROCESSED_MD_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    print(f"- {d}")

# Force directory listing to refresh Drive cache
pdf_files = sorted([f for f in RAW_PDF_DIR.iterdir()
                    if f.is_file() and f.suffix.lower() == ".pdf"])
print(f"\nFound {len(pdf_files)} PDF files:")
for f in pdf_files:
    print(f"   - {f.name}  ({f.stat().st_size / 1024:.0f} KB)")

if not pdf_files:
    print("\nNo PDFs found! Upload your manuals to data/raw/manuals/")

- /content/drive/MyDrive/RAG_Atlas200DK/data/raw/manuals
- /content/drive/MyDrive/RAG_Atlas200DK/data/processed/markdown
- /content/drive/MyDrive/RAG_Atlas200DK/data/output

Found 10 PDF files:
   - ATCToolInstructions.pdf  (1278 KB)
   - ApplicationDevelopmentGuide(MindStudio).pdf  (964 KB)
   - Atlas200AIAcceleratorModuleApplicationSoftwareDevelopmentGuide(Models3000) .pdf  (5329 KB)
   - EnvironmentDeploymentGuide.pdf  (4726 KB)
   - Huawei Atlas 200 DK developer kit Technical White Paper (Model 3000) 08.pdf  (2037 KB)
   - MindStudioInstallationGuideUbuntuX86.pdf  (1075 KB)
   - TBECustomOperatorDevelopmentGuide(CLI).pdf  (652 KB)
   - TensorFlowNetworkModelPortingandTrainingGuide.pdf  (2353 KB)
   - UserGuide2020.pdf  (1672 KB)
   - UserGuide2021.pdf  (4554 KB)


In [32]:
import os

base_path = str(BASE_DIR)
print(f"Analyzing contents of: {base_path}\n")

if not os.path.exists(base_path):
    print("Main folder not found!")
else:
    for root, dirs, files in os.walk(base_path):
        # Skip hidden directories and output dirs to keep it clean
        dirs[:] = [d for d in dirs if not d.startswith(".")]
        level = root.replace(base_path, "").count(os.sep)
        indent = " " * 4 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = " " * 4 * (level + 1)
        for f in files:
            print(f"{subindent}{f}")

Analyzing contents of: /content/drive/MyDrive/RAG_Atlas200DK

RAG_Atlas200DK/
    data_chunking.ipynb
    data/
        raw/
            manuals/
                ATCToolInstructions.pdf
                ApplicationDevelopmentGuide(MindStudio).pdf
                Atlas200AIAcceleratorModuleApplicationSoftwareDevelopmentGuide(Models3000) .pdf
                EnvironmentDeploymentGuide.pdf
                Huawei Atlas 200 DK developer kit Technical White Paper (Model 3000) 08.pdf
                MindStudioInstallationGuideUbuntuX86.pdf
                TBECustomOperatorDevelopmentGuide(CLI).pdf
                TensorFlowNetworkModelPortingandTrainingGuide.pdf
                UserGuide2020.pdf
                UserGuide2021.pdf
        processed/
            markdown/
        output/
            processed_markdown.zip
            chunking_output.zip
    output/
        processed_markdown.zip
        chunking_output.zip


In [33]:
# ============================================================================
# Cell A.6 — Tokenizer Configuration
# ============================================================================
TIKTOKEN_ENCODING = "cl100k_base"
enc = tiktoken.get_encoding(TIKTOKEN_ENCODING)


def count_tokens(text: str) -> int:
    """Count BPE tokens using tiktoken."""
    return len(enc.encode(text))


# Quick validation
_test_count = count_tokens("Hello, Ascend 310 AI Processor with DaVinci cores!")
assert _test_count > 0, "Tokenizer returned 0 tokens"
print(f"Tokenizer ready: {TIKTOKEN_ENCODING} (test: {_test_count} tokens)")

Tokenizer ready: cl100k_base (test: 14 tokens)


---
## 📄 Seção B: Extração de Documentos PDF → Markdown

*Parsing layout-aware dos manuais técnicos Huawei usando `marker-pdf` com pipeline de pós-processamento para:*
- **Remoção de cabeçalhos/rodapés** recorrentes (ex.: "Huawei Proprietary", banners de capítulo)
- **Correção de hifenização** em termos técnicos quebrados entre linhas (ex.: `DaV-inci` → `DaVinci`)
- **Normalização de unidades** de hardware (`TFLOPS`, `TOPS`, `GB/s`, `mA`, `V`)

In [34]:
# ============================================================================
# Cell B.1 — Post-Processing Pipeline for Huawei Manuals
# ============================================================================

# Huawei proprietary footers / headers to strip
_HEADER_FOOTER_PATTERNS = [
    re.compile(r"(?im)^.*Huawei\s+Proprietary.*$"),
    re.compile(r"(?im)^.*Huawei\s+Confidential.*$"),
    re.compile(r"(?im)^.*Huawei\s+Technologies\s+Co\.?,?\s*Ltd\.?.*$"),
    re.compile(r"(?im)^\s*Issue\s+\d+.*Date.*$"),
    re.compile(r"(?im)^\s*Copyright\s+©.*Huawei.*$"),
    re.compile(r"(?im)^\s*Page\s+\d+\s*$"),
    re.compile(r"(?im)^\s*\d+\s*$"),  # bare page numbers
]

# Technical terms commonly broken by PDF line wraps
_HYPHEN_REJOINS = [
    (re.compile(r"DaV-\s*inci", re.I), "DaVinci"),
    (re.compile(r"NC1HW-\s*C0", re.I), "NC1HWC0"),
    (re.compile(r"conv2-\s*d", re.I), "conv2d"),
    (re.compile(r"Ascend-\s*CL", re.I), "AscendCL"),
    (re.compile(r"Mind-\s*Studio", re.I), "MindStudio"),
    (re.compile(r"Tensor-\s*Flow", re.I), "TensorFlow"),
    (re.compile(r"FRACTAL-\s*Z", re.I), "FRACTAL_Z"),
    (re.compile(r"LPDDR-?\s*4X", re.I), "LPDDR4X"),
    # Generic: rejoin word-hyphen-newline-word
    (re.compile(r"(\w)-\n(\w)"), r"\1\2"),
]

# Unit normalization: ensure consistent spacing/casing
_UNIT_NORMALIZATIONS = [
    (re.compile(r"(?<!\w)(T\s*F\s*L\s*O\s*P\s*S)(?!\w)", re.I), "TFLOPS"),
    (re.compile(r"(?<!\w)(T\s*O\s*P\s*S)(?!\w)", re.I), "TOPS"),
    (re.compile(r"(\d)\s*(GB\s*/\s*s)", re.I), r"\1 GB/s"),
    (re.compile(r"(\d)\s*(mA)(?!\w)", re.I), r"\1 mA"),
    (re.compile(r"(\d)\s*([VW])(?!\w)"), r"\1 \2"),
]


def postprocess_markdown(text: str) -> str:
    """Apply Huawei-specific post-processing to extracted Markdown."""
    # Strip headers/footers
    for pat in _HEADER_FOOTER_PATTERNS:
        text = pat.sub("", text)

    # Fix broken technical terms
    for pat, repl in _HYPHEN_REJOINS:
        text = pat.sub(repl, text)

    # Normalize hardware units
    for pat, repl in _UNIT_NORMALIZATIONS:
        text = pat.sub(repl, text)

    # Collapse excessive blank lines (more than 2 → 2)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# Quick test
_test_in = "DaV-\ninci core at 8 T F L O P S\nHuawei Proprietary and Confidential"
_test_out = postprocess_markdown(_test_in)
print("Post-processing pipeline ready")
print(f'   Test input:  "{_test_in[:50]}..."')
print(f'   Test output: "{_test_out[:60]}..."')

✅ Post-processing pipeline ready
   Test input:  "DaV-
inci core at 8 T F L O P S
Huawei Proprietary..."
   Test output: "DaVinci core at 8 TFLOPS..."


In [35]:
# ============================================================================
# Cell B.2 — Batch PDF to Markdown Converter
# ============================================================================


def convert_pdfs_to_markdown(input_dir: Path, output_dir: Path) -> dict:
    """
    Convert all PDFs in input_dir to cleaned Markdown files in output_dir.

    Uses marker-pdf Python API with disable_ocr=True, which instructs the
    ConfigParser to select a converter class that performs pure text-layer
    extraction via pdftext. No Docker, vLLM, or llama-server required.

    This works because the Huawei manuals are digital PDFs with embedded
    text layers (not scanned documents).

    Returns:
        dict mapping doc_name -> {md_path, tokens, chars, status}
    """
    from marker.config.parser import ConfigParser
    from marker.models import create_model_dict
    from marker.output import save_output

    # Force directory listing to refresh Drive cache
    all_files = list(input_dir.iterdir())
    pdf_files_list = sorted(
        f for f in all_files if f.is_file() and f.suffix.lower() == ".pdf"
    )
    if not pdf_files_list:
        raise FileNotFoundError(f"No PDF files found in {input_dir}")

    print(f"Found {len(pdf_files_list)} PDFs to convert")

    # disable_ocr=True: pure text-layer extraction, no VLM inference server
    cli_options = {
        "output_format": "markdown",
        "disable_ocr": True,
        "disable_image_extraction": True,
        "paginate_output": False,
    }
    config_parser = ConfigParser(cli_options)

    print("Initializing marker-pdf models (disable_ocr mode, no Docker)...")
    converter_cls = config_parser.get_converter_cls()
    config_dict = config_parser.generate_config_dict()
    config_dict["pdftext_workers"] = 1  # avoid nested multiprocessing

    artifact_dict = create_model_dict()
    converter = converter_cls(
        config=config_dict,
        artifact_dict=artifact_dict,
        processor_list=config_parser.get_processors(),
        renderer=config_parser.get_renderer(),
    )
    print("Models loaded\n")

    results = {}
    for pdf_path in tqdm(pdf_files_list, desc="Converting PDFs", unit="file"):
        doc_name = pdf_path.stem.strip()
        md_filename = re.sub(r"[^\w\-.]", "_", doc_name) + ".md"
        md_out_path = output_dir / md_filename

        try:
            rendered = converter(str(pdf_path))
            raw_md = rendered.markdown

            # Apply Huawei-specific post-processing
            clean_md = postprocess_markdown(raw_md)

            md_out_path.write_text(clean_md, encoding="utf-8")
            token_count = count_tokens(clean_md)

            results[doc_name] = {
                "md_path": str(md_out_path),
                "tokens": token_count,
                "chars": len(clean_md),
                "status": "OK"
            }
            print(f"  OK {doc_name}: {token_count:,} tokens, {len(clean_md):,} chars")

        except Exception as e:
            results[doc_name] = {
                "md_path": None,
                "tokens": 0,
                "chars": 0,
                "status": f"FAIL {type(e).__name__}: {e}"
            }
            import traceback; traceback.print_exc()
            print(f"  FAIL {doc_name}: {e}")

    return results

### ▶️ Execução da Conversão em Lote

*Conversão de todos os PDFs encontrados em `data/raw/manuals/`. O modelo de extração é carregado uma única vez e reutilizado para todos os documentos, otimizando o uso de memória GPU.*

In [36]:
# ============================================================================
# Cell B.3 — Execute Batch Conversion
# ============================================================================
conversion_results = convert_pdfs_to_markdown(RAW_PDF_DIR, PROCESSED_MD_DIR)

# Summary table
print("\n" + "=" * 72)
print(f"{'Document':<45} {'Tokens':>8}  {'Chars':>8}  Status")
print("=" * 72)
total_tokens = 0
for name, info in sorted(conversion_results.items()):
    total_tokens += info["tokens"]
    print(f"{name[:44]:<45} {info['tokens']:>8,}  {info['chars']:>8,}  {info['status']}")
print("=" * 72)
print(f"{'TOTAL':<45} {total_tokens:>8,}")

Found 10 PDFs to convert
Initializing marker-pdf models (disable_ocr mode, no Docker)...
Models loaded



Converting PDFs:   0%|          | 0/10 [00:00<?, ?file/s]

2026-09-28 13:32:53,839 [INFO] marker: Table processing stats: {'tables_pdftext': 86, 'tables_total': 141}
INFO:marker:Table processing stats: {'tables_pdftext': 86, 'tables_total': 141}


  OK ATCToolInstructions: 37,719 tokens, 160,478 chars


2026-09-28 13:33:12,247 [INFO] marker: Table processing stats: {'tables_pdftext': 96, 'tables_total': 14}
INFO:marker:Table processing stats: {'tables_pdftext': 96, 'tables_total': 14}


...

  OK TensorFlowNetworkModelPortingandTrainingGuide: 76,327 tokens, 353,217 chars


2026-09-28 13:37:55,217 [INFO] marker: Table processing stats: {'tables_pdftext': 724, 'tables_total': 30}
INFO:marker:Table processing stats: {'tables_pdftext': 724, 'tables_total': 30}


  OK UserGuide2020: 20,524 tokens, 79,970 chars


2026-09-28 13:38:18,551 [INFO] marker: Table processing stats: {'tables_pdftext': 737, 'tables_total': 38}
INFO:marker:Table processing stats: {'tables_pdftext': 737, 'tables_total': 38}


  OK UserGuide2021: 29,957 tokens, 117,392 chars

Document                                        Tokens     Chars  Status
ATCToolInstructions                             37,719   160,478  OK
ApplicationDevelopmentGuide(MindStudio)         25,311   108,117  OK
Atlas200AIAcceleratorModuleApplicationSoftwa   246,233  1,060,979  OK
EnvironmentDeploymentGuide                      30,161   117,544  OK
Huawei Atlas 200 DK developer kit Technical      6,228    26,385  OK
MindStudioInstallationGuideUbuntuX86            19,606    88,690  OK
TBECustomOperatorDevelopmentGuide(CLI)          19,123    84,728  OK
TensorFlowNetworkModelPortingandTrainingGuid    76,327   353,217  OK
UserGuide2020                                   20,524    79,970  OK
UserGuide2021                                   29,957   117,392  OK
TOTAL                                          511,189


In [37]:
# ============================================================================
# Cell B.4 — Markdown Preview Utility
# ============================================================================

def preview_markdown_sample(md_file_path: str, char_limit: int = 2000):
    """Display raw text and rendered Markdown preview inside Colab/Jupyter."""
    text = Path(md_file_path).read_text(encoding="utf-8")
    sample = text[:char_limit]
    token_count = count_tokens(text)

    print(f"   File: {Path(md_file_path).name}")
    print(f"   Total: {len(text):,} chars | {token_count:,} tokens")
    print(f"   Showing first {char_limit:,} characters\n")

    # Raw text view
    print("─── Raw Text ───")
    print(sample)
    if len(text) > char_limit:
        print(f"\n... ({len(text) - char_limit:,} more characters)")
    print("─" * 50)

    # Rendered Markdown view
    display(IPyMarkdown(f"**── Rendered Preview ──**\n\n{sample}"))

In [38]:
# ============================================================================
# Cell B.5 — Preview First Converted Document
# ============================================================================
md_files = sorted(PROCESSED_MD_DIR.glob("*.md"))
if md_files:
    preview_markdown_sample(str(md_files[0]), char_limit=2000)
else:
    print("No Markdown files found — run conversion cell first")

📄 File: ATCToolInstructions.md
   Total: 160,478 chars | 37,719 tokens
   Showing first 2,000 characters

─── Raw Text ───
# **Atlas 200 DK V100R020C00 ATC Tool Instructions**

**Issue** 01 **Date** 2021-01-29

# **Trademarks and Permissions**

# **Notice**

The purchased products, services and features are stipulated by the contract made between Huawei and the customer. All or part of the products, services and features described in this document may not be within the purchase scope or the usage scope. Unless otherwise specified in the contract, all statements, information, and recommendations in this document are provided "AS IS" without warranties, guarantees or representations of any kind, either express or implied.

The information in this document is subject to change without notice. Every effort has been made in the preparation of this document to ensure accuracy of the contents, but all statements, information, and recommendations in this document do not constitute a warranty o

**── Rendered Preview ──**

# **Atlas 200 DK V100R020C00 ATC Tool Instructions**

**Issue** 01 **Date** 2021-01-29

# **Trademarks and Permissions**

# **Notice**

The purchased products, services and features are stipulated by the contract made between Huawei and the customer. All or part of the products, services and features described in this document may not be within the purchase scope or the usage scope. Unless otherwise specified in the contract, all statements, information, and recommendations in this document are provided "AS IS" without warranties, guarantees or representations of any kind, either express or implied.

The information in this document is subject to change without notice. Every effort has been made in the preparation of this document to ensure accuracy of the contents, but all statements, information, and recommendations in this document do not constitute a warranty of any kind, express or implied.

# **Contents**

| 1 Introduction..............................................................................................................................                                                   | 1   |
|------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|-----|
| 4.1 Overview.................................................................................................................................................................................. | 38  |
| 4.3 CSC Configuration................................................................................................................................................................          | 43  |
| 4.4 Cropping and Padding Configuration............................................................................................................................                             | 60  |
| 5 Single-Operator JSON File Configuration..........................

In [39]:
# ============================================================================
# Cell B.6 — Table Integrity Auditor
# ============================================================================

def audit_table_integrity(md_file_path: str) -> dict:
    """
    Scan a Markdown file for pipe tables and check for truncation/misalignment.

    Detects:
    - Inconsistent column counts within a table block
    - Single-row tables (possible truncation)
    - Missing separator rows (|---|---|)

    Returns:
        dict with 'file', 'tables_found', 'issues', and 'details' keys.
    """
    text = Path(md_file_path).read_text(encoding="utf-8")
    lines = text.split("\n")

    # Detect table blocks: consecutive lines containing pipes
    table_blocks = []
    current_block = []
    for i, line in enumerate(lines):
        stripped = line.strip()
        if stripped.startswith("|") and "|" in stripped[1:]:
            current_block.append((i + 1, stripped))
        else:
            if current_block:
                table_blocks.append(current_block)
                current_block = []
    if current_block:
        table_blocks.append(current_block)

    issues = []
    for idx, block in enumerate(table_blocks):
        col_counts = [row[1].count("|") for row in block]

        # Check column consistency
        if len(set(col_counts)) > 1:
            issues.append(
                f"Table #{idx+1} (lines {block[0][0]}-{block[-1][0]}): "
                f"inconsistent column counts {sorted(set(col_counts))}"
            )

        # Check for truncated single-row tables
        if len(block) < 2:
            issues.append(
                f"Table #{idx+1} (line {block[0][0]}): "
                f"single-row table — possible truncation"
            )

        # Check for separator row
        has_separator = any(
            re.match(r"^\|[\s\-:|]+\|$", row[1]) for row in block
        )
        if not has_separator and len(block) > 1:
            issues.append(
                f"Table #{idx+1} (lines {block[0][0]}-{block[-1][0]}): "
                f"missing separator row (|---|---|)"
            )

    result = {
        "file": Path(md_file_path).name,
        "tables_found": len(table_blocks),
        "issues": len(issues),
        "details": issues,
    }

    print(f"- {result['file']}: {result['tables_found']} tables, "
          f"{result['issues']} issues")
    for issue in issues[:15]:  # cap verbose output
        print(f" ======== Error:  {issue}")

    return result

In [40]:
# ============================================================================
# Cell B.7 — Run Table Integrity Audit on All Converted Files
# ============================================================================
print("--- Table Integrity Audit\n")
all_audits = {}
for md_file in sorted(PROCESSED_MD_DIR.glob("*.md")):
    all_audits[md_file.name] = audit_table_integrity(str(md_file))
    print()

total_tables = sum(a["tables_found"] for a in all_audits.values())
total_issues = sum(a["issues"] for a in all_audits.values())
print(f"{'=' * 55}")
print(f"Summary: {total_tables} tables across {len(all_audits)} files, "
      f"{total_issues} issues detected")

🔍 Table Integrity Audit

📊 ATCToolInstructions.md: 86 tables, 0 issues

📊 ApplicationDevelopmentGuide_MindStudio_.md: 10 tables, 0 issues

📊 Atlas200AIAcceleratorModuleApplicationSoftwareDevelopmentGuide_Models3000_.md: 467 tables, 0 issues

📊 EnvironmentDeploymentGuide.md: 12 tables, 0 issues

📊 Huawei_Atlas_200_DK_developer_kit_Technical_White_Paper__Model_3000__08.md: 15 tables, 0 issues

📊 MindStudioInstallationGuideUbuntuX86.md: 13 tables, 0 issues

📊 TBECustomOperatorDevelopmentGuide_CLI_.md: 13 tables, 0 issues

📊 TensorFlowNetworkModelPortingandTrainingGuide.md: 100 tables, 0 issues

📊 UserGuide2020.md: 8 tables, 0 issues

📊 UserGuide2021.md: 13 tables, 0 issues

Summary: 737 tables across 10 files, 0 issues detected


In [41]:
# ============================================================================
# Cell B.8 — Export Processed Markdown Artifacts
# ============================================================================
export_artifact(PROCESSED_MD_DIR, "processed_markdown.zip")

Created /content/drive/MyDrive/RAG_Atlas200DK/data/output/processed_markdown.zip (444 KB)
Copied to Google Drive: /content/drive/MyDrive/RAG_Atlas200DK/output/processed_markdown.zip


---
## 🧪 Seção C: Laboratório Comparativo de Estratégias de Chunking

*Avaliação sistemática de três abordagens de chunking para RAG, aplicadas sobre os documentos Markdown extraídos:*

| # | Estratégia | Descrição |
|---|-----------|----------|
| 1 | **Recursive Window** (baseline) | Splitting recursivo por caracteres com `RecursiveCharacterTextSplitter` |
| 2 | **Markdown Header** (hierárquico) | Splitting por cabeçalhos `#`/`##`/`###` com preservação de hierarquia |
| 3 | **Atomic Table-Aware** (híbrido) | Chunking atômico preservando tabelas intactas + breadcrumbs de contexto |

In [42]:
# ============================================================================
# Cell C.1 — Strategy 1: Fixed/Recursive Window Splitting (Baseline)
# ============================================================================
from langchain_text_splitters import RecursiveCharacterTextSplitter


def strategy_recursive_split(
    text: str,
    chunk_size: int = 1024,
    chunk_overlap: int = 128,
) -> list[dict]:
    """
    Strategy 1: Fixed/recursive window splitting (baseline).

    Uses RecursiveCharacterTextSplitter with tiktoken-based length function.
    Splits preferring paragraph breaks, then line breaks, then sentences.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=count_tokens,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_text(text)
    return [
        {
            "text": chunk,
            "strategy": "recursive",
            "chunk_index": i,
            "approx_tokens": count_tokens(chunk),
            "metadata": {
                "chunk_size_param": chunk_size,
                "chunk_overlap_param": chunk_overlap,
            }
        }
        for i, chunk in enumerate(chunks)
    ]


print("Strategy 1 (Recursive Window) defined")

✅ Strategy 1 (Recursive Window) defined


### Estratégia 2: Splitting por Cabeçalhos Markdown

*Preserva a hierarquia de seções como metadados de cada chunk. Por exemplo, um trecho sob `## Supported Operators > ### Caffe Network Model` carrega essa informação de escopo. Seções que excedem o limite de tokens são subdivididas com `RecursiveCharacterTextSplitter`.*

In [43]:
# ============================================================================
# Cell C.2 — Strategy 2: Markdown Structural / Header Splitting
# ============================================================================
from langchain_text_splitters import MarkdownHeaderTextSplitter


def strategy_markdown_header_split(
    text: str,
    max_chunk_tokens: int = 1024,
) -> list[dict]:
    """
    Strategy 2: Markdown structural / header splitting.

    Splits along header boundaries (#, ##, ###, ####) and preserves the
    header hierarchy as metadata. Oversized sections are sub-split with
    RecursiveCharacterTextSplitter.
    """
    headers_to_split_on = [
        ("#",    "Header_1"),
        ("##",   "Header_2"),
        ("###",  "Header_3"),
        ("####", "Header_4"),
    ]

    md_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on,
        strip_headers=False,
    )
    header_chunks = md_splitter.split_text(text)

    # Sub-split oversized sections
    sub_splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_chunk_tokens,
        chunk_overlap=64,
        length_function=count_tokens,
    )

    results = []
    for doc in header_chunks:
        content = doc.page_content
        metadata = dict(doc.metadata)
        header_path = " > ".join(
            v for k, v in sorted(metadata.items())
            if k.startswith("Header_")
        )

        if count_tokens(content) > max_chunk_tokens:
            sub_chunks = sub_splitter.split_text(content)
            for j, sc in enumerate(sub_chunks):
                results.append({
                    "text": sc,
                    "strategy": "markdown_header",
                    "chunk_index": len(results),
                    "approx_tokens": count_tokens(sc),
                    "metadata": {
                        "header_path": header_path,
                        **metadata,
                        "sub_chunk": j,
                    }
                })
        else:
            results.append({
                "text": content,
                "strategy": "markdown_header",
                "chunk_index": len(results),
                "approx_tokens": count_tokens(content),
                "metadata": {
                    "header_path": header_path,
                    **metadata,
                }
            })

    return results


print("Strategy 2 (Markdown Header) defined")

✅ Strategy 2 (Markdown Header) defined


### Estratégia 3: Chunking Atômico com Preservação de Tabelas e Breadcrumbs

*Abordagem híbrida customizada que:*
1. **Detecta blocos atômicos** — tabelas pipe de Markdown e blocos de código são mantidos intactos (sem fragmentação entre linhas)
2. **Injeta breadcrumbs** — cada chunk recebe um prefixo `[Doc: X > Section: Y > Sub: Z]` com o caminho hierárquico completo
3. **Enriquece metadados** — `doc_name`, `header_path`, `is_table`, `is_code_block`, `approx_tokens`

In [44]:
# ============================================================================
# Cell C.3 — Strategy 3: Atomic Table-Aware & Breadcrumb Hybrid Chunking
# ============================================================================

def strategy_atomic_table_aware(
    text: str,
    doc_name: str = "unknown",
    max_chunk_tokens: int = 1024,
    chunk_overlap: int = 64,
) -> list[dict]:
    """
    Strategy 3: Atomic table-aware chunking with breadcrumb context.

    Pipeline:
    1. Parse the heading hierarchy to build section paths.
    2. Within each section, identify atomic blocks (tables, code fences).
    3. Keep atomic blocks intact; sub-split prose exceeding max_chunk_tokens.
    4. Prepend breadcrumb path to every chunk for standalone readability.

    Returns:
        List of chunk dicts with text, strategy, chunk_index, approx_tokens, metadata.
    """

    # --- Step 1: Parse header hierarchy into sections ---
    lines = text.split("\n")
    sections = []  # list of (header_path_str, content_text, has_table_bool)
    current_headers = {}  # level (int) → header text (str)
    current_lines = []
    current_has_table = False

    def _flush_section():
        nonlocal current_lines, current_has_table
        if current_lines:
            path_parts = [
                current_headers[k]
                for k in sorted(current_headers.keys())
            ]
            sections.append((
                " > ".join(path_parts) if path_parts else "Root",
                "\n".join(current_lines),
                current_has_table,
            ))
        current_lines = []
        current_has_table = False

    for line in lines:
        header_match = re.match(r"^(#{1,4})\s+(.+)$", line)
        if header_match:
            _flush_section()
            level = len(header_match.group(1))
            title = header_match.group(2).strip()
            current_headers[level] = title
            # Clear deeper headers when a higher-level header appears
            for k in list(current_headers.keys()):
                if k > level:
                    del current_headers[k]
            current_lines.append(line)
        else:
            current_lines.append(line)
            if line.strip().startswith("|"):
                current_has_table = True
    _flush_section()

    # --- Step 2: Chunk each section preserving atomic blocks ---
    results = []

    for header_path, section_text, _ in sections:
        breadcrumb = f"[Doc: {doc_name} > {header_path}]"

        # Split section into atomic (tables, code) vs prose segments
        # Pattern matches: code fences OR consecutive table rows (2+ lines)
        parts = re.split(
            r"(```[\s\S]*?```|(?:^\|.+\|$\n?){2,})",
            section_text,
            flags=re.MULTILINE,
        )

        for part in parts:
            part_stripped = part.strip()
            if not part_stripped:
                continue

            is_table = part_stripped.startswith("|")
            is_code = part_stripped.startswith("```")
            is_atomic = is_table or is_code

            chunk_text = f"{breadcrumb}\n\n{part_stripped}"
            tok_count = count_tokens(chunk_text)

            if is_atomic or tok_count <= max_chunk_tokens:
                # Keep atomic blocks intact regardless of size
                results.append({
                    "text": chunk_text,
                    "strategy": "atomic_table_aware",
                    "chunk_index": len(results),
                    "approx_tokens": tok_count,
                    "metadata": {
                        "doc_name": doc_name,
                        "header_path": header_path,
                        "is_table": is_table,
                        "is_code_block": is_code,
                    }
                })
            else:
                # Sub-split oversized prose with breadcrumb prepended
                sub_splitter = RecursiveCharacterTextSplitter(
                    chunk_size=max_chunk_tokens,
                    chunk_overlap=chunk_overlap,
                    length_function=count_tokens,
                )
                for sc in sub_splitter.split_text(part_stripped):
                    sc_with_breadcrumb = f"{breadcrumb}\n\n{sc}"
                    results.append({
                        "text": sc_with_breadcrumb,
                        "strategy": "atomic_table_aware",
                        "chunk_index": len(results),
                        "approx_tokens": count_tokens(sc_with_breadcrumb),
                        "metadata": {
                            "doc_name": doc_name,
                            "header_path": header_path,
                            "is_table": False,
                            "is_code_block": False,
                        }
                    })

    return results


print("Strategy 3 (Atomic Table-Aware) defined")

✅ Strategy 3 (Atomic Table-Aware) defined


In [45]:
# ============================================================================
# Cell C.4 — Chunking Strategy Comparison Engine
# ============================================================================

def compare_chunking_strategies(
    sample_md_text: str,
    doc_name: str = "sample",
    chunk_size: int = 1024,
    chunk_overlap: int = 128,
) -> dict:
    """
    Run all three chunking strategies on the same text and compare results.

    Returns:
        dict mapping strategy_name → {chunks, total_chunks,
        token_min/max/mean/median, table_chunks}
    """
    strategies = {
        "1_recursive": lambda: strategy_recursive_split(
            sample_md_text, chunk_size=chunk_size, chunk_overlap=chunk_overlap
        ),
        "2_markdown_header": lambda: strategy_markdown_header_split(
            sample_md_text, max_chunk_tokens=chunk_size
        ),
        "3_atomic_table_aware": lambda: strategy_atomic_table_aware(
            sample_md_text, doc_name=doc_name,
            max_chunk_tokens=chunk_size, chunk_overlap=chunk_overlap
        ),
    }

    results = {}
    for name, fn in strategies.items():
        chunks = fn()
        token_counts = [c["approx_tokens"] for c in chunks]
        table_chunks = sum(
            1 for c in chunks
            if c.get("metadata", {}).get("is_table", False)
        )

        results[name] = {
            "chunks": chunks,
            "total_chunks": len(chunks),
            "token_min": min(token_counts) if token_counts else 0,
            "token_max": max(token_counts) if token_counts else 0,
            "token_mean": (
                sum(token_counts) / len(token_counts) if token_counts else 0
            ),
            "token_median": (
                sorted(token_counts)[len(token_counts) // 2]
                if token_counts else 0
            ),
            "table_chunks": table_chunks,
        }

    return results


print("Comparison engine defined")

✅ Comparison engine defined


In [46]:
# ============================================================================
# Cell C.5 — Comparison Visualization & Reporting
# ============================================================================

def display_comparison_report(results: dict, sample_chunks: int = 2):
    """Display a side-by-side comparison of chunking strategy results."""

    # ── Summary table ──
    print("\n" + "=" * 85)
    print(f"{'Strategy':<28} {'Chunks':>7} {'Min':>6} {'Max':>6} "
          f"{'Mean':>7} {'Median':>7} {'Tables':>7}")
    print("=" * 85)
    for name, stats in results.items():
        print(
            f"{name:<28} {stats['total_chunks']:>7} "
            f"{stats['token_min']:>6} {stats['token_max']:>6} "
            f"{stats['token_mean']:>7.0f} {stats['token_median']:>7} "
            f"{stats['table_chunks']:>7}"
        )
    print("=" * 85)

    # ── Token distribution histogram (text-based) ──
    buckets = [0, 64, 128, 256, 512, 768, 1024, 1536, 2048, float("inf")]
    for name, stats in results.items():
        token_counts = [c["approx_tokens"] for c in stats["chunks"]]
        print(f"\n --- Token Distribution — {name}")
        for i in range(len(buckets) - 1):
            lo, hi = buckets[i], buckets[i + 1]
            count = sum(1 for t in token_counts if lo <= t < hi)
            bar = "█" * min(count, 60)  # cap bar width
            if hi != float("inf"):
                label = f"{lo:>5}-{hi:<5}"
            else:
                label = f"{lo:>5}+    "
            print(f"  {label} │ {bar} ({count})")

    # ── Sample chunk previews ──
    for name, stats in results.items():
        print(f"\n{'─' * 80}")
        print(f"Sample Chunks — {name}")
        print(f"{'─' * 80}")
        for chunk in stats["chunks"][:sample_chunks]:
            preview = chunk["text"][:400]
            meta = chunk.get("metadata", {})
            print(f"\n  [Chunk #{chunk['chunk_index']}] "
                  f"({chunk['approx_tokens']} tokens)")
            if "header_path" in meta:
                print(f"  Path: {meta['header_path']}")
            if meta.get("is_table"):
                print("  TABLE chunk")
            print(f"  {'.' * 60}")
            print(textwrap.indent(preview, "  "))
            if len(chunk["text"]) > 400:
                print(f"  ... ({len(chunk['text']) - 400} more chars)")


print("✅ Visualization functions defined")

✅ Visualization functions defined


### ▶️ Execução do Laboratório Comparativo

*Selecionamos automaticamente um documento que contenha tabelas densas (ex.: `ATCToolInstructions`) para avaliar as três estratégias em um cenário realista com parâmetros técnicos, especificações de hardware e listas de operadores.*

In [47]:
# ============================================================================
# Cell C.6 — Run Comparison on a Sample Document
# ============================================================================

# Select a document likely to contain dense parameter tables
_target_keywords = ["ATCToolInstructions", "Atlas200AI", "WhitePaper", "Technical"]
sample_md_path = None

for md_file in sorted(PROCESSED_MD_DIR.glob("*.md")):
    if any(kw.lower() in md_file.stem.lower() for kw in _target_keywords):
        sample_md_path = md_file
        break

# Fallback to first available file
if sample_md_path is None:
    _all_md = sorted(PROCESSED_MD_DIR.glob("*.md"))
    if _all_md:
        sample_md_path = _all_md[0]

if sample_md_path:
    sample_text = sample_md_path.read_text(encoding="utf-8")
    print(f"--- Selected document: {sample_md_path.name}")
    print(f"   Size: {len(sample_text):,} chars | {count_tokens(sample_text):,} tokens\n")

    comparison_results = compare_chunking_strategies(
        sample_text,
        doc_name=sample_md_path.stem,
        chunk_size=1024,
        chunk_overlap=128,
    )
    display_comparison_report(comparison_results, sample_chunks=3)
else:
    print("No Markdown files found — run Section B first")

📄 Selected document: ATCToolInstructions.md
   Size: 160,478 chars | 37,719 tokens


Strategy                      Chunks    Min    Max    Mean  Median  Tables
1_recursive                       44    323   1015     914     955       0
2_markdown_header                174      4   1013     223      87       0
3_atomic_table_aware             259     16   1982     176     116      86

📊 Token Distribution — 1_recursive
      0-64    │  (0)
     64-128   │  (0)
    128-256   │  (0)
    256-512   │ █ (1)
    512-768   │ ███ (3)
    768-1024  │ ████████████████████████████████████████ (40)
   1024-1536  │  (0)
   1536-2048  │  (0)
   2048+     │  (0)

📊 Token Distribution — 2_markdown_header
      0-64    │ ████████████████████████████████████████████████████████████ (82)
     64-128   │ ████████████████ (16)
    128-256   │ ██████████████████████████████ (30)
    256-512   │ █████████████████████ (21)
    512-768   │ █████ (5)
    768-1024  │ ████████████████████ (20)
   1024-1536  │  (0)


### 📐 Varredura de Parâmetros

*Execução das três estratégias com diferentes combinações de `chunk_size` e `chunk_overlap` para avaliar a sensibilidade dos resultados aos hiperparâmetros de chunking.*

In [48]:
# ============================================================================
# Cell C.7 — Parameter Sweep Across Chunk Sizes
# ============================================================================
if sample_md_path:
    print("Parameter Sweep: chunk_size × chunk_overlap sensitivity\n")

    sweep_summary = []
    for cs in [512, 1024]:
        for co in [64, 128]:
            print(f"{'=' * 65}")
            print(f"  chunk_size={cs}, chunk_overlap={co}")
            print(f"{'=' * 65}")

            r = compare_chunking_strategies(
                sample_text,
                doc_name=sample_md_path.stem,
                chunk_size=cs,
                chunk_overlap=co,
            )

            for sname, stats in r.items():
                row = {
                    "chunk_size": cs,
                    "chunk_overlap": co,
                    "strategy": sname,
                    "n_chunks": stats["total_chunks"],
                    "mean_tokens": round(stats["token_mean"]),
                    "table_chunks": stats["table_chunks"],
                }
                sweep_summary.append(row)
                print(
                    f"    {sname:<28}: {stats['total_chunks']:>4} chunks, "
                    f"mean={stats['token_mean']:.0f} tok, "
                    f"tables={stats['table_chunks']}"
                )
            print()

    # Save sweep results
    sweep_path = OUTPUT_DIR / "parameter_sweep.json"
    sweep_path.write_text(
        json.dumps(sweep_summary, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    print(f"\n Sweep results saved to {sweep_path}")
else:
    print("No sample document — run Section B first")

📐 Parameter Sweep: chunk_size × chunk_overlap sensitivity

  chunk_size=512, chunk_overlap=64
    1_recursive                 :   96 chunks, mean=417 tok, tables=0
    2_markdown_header           :  204 chunks, mean=197 tok, tables=0
    3_atomic_table_aware        :  272 chunks, mean=177 tok, tables=86

  chunk_size=512, chunk_overlap=128
    1_recursive                 :  101 chunks, mean=430 tok, tables=0
    2_markdown_header           :  204 chunks, mean=197 tok, tables=0
    3_atomic_table_aware        :  272 chunks, mean=179 tok, tables=86

  chunk_size=1024, chunk_overlap=64
    1_recursive                 :   42 chunks, mean=917 tok, tables=0
    2_markdown_header           :  174 chunks, mean=223 tok, tables=0
    3_atomic_table_aware        :  259 chunks, mean=176 tok, tables=86

  chunk_size=1024, chunk_overlap=128
    1_recursive                 :   44 chunks, mean=914 tok, tables=0
    2_markdown_header           :  174 chunks, mean=223 tok, tables=0
    3_atomic_table_aw

In [49]:
# ============================================================================
# Cell C.8 — Export Chunks to JSON
# ============================================================================

def export_chunks_to_json(
    results: dict,
    output_path: Path,
    doc_name: str,
) -> Path:
    """Export all strategy results to a structured JSON file."""
    export_data = {
        "doc_name": doc_name,
        "encoding": TIKTOKEN_ENCODING,
        "strategies": {},
    }

    for strategy_name, stats in results.items():
        export_data["strategies"][strategy_name] = {
            "summary": {
                k: v for k, v in stats.items() if k != "chunks"
            },
            "chunks": [
                {
                    "text": c["text"],
                    "approx_tokens": c["approx_tokens"],
                    "metadata": c.get("metadata", {}),
                }
                for c in stats["chunks"]
            ],
        }

    json_path = output_path / f"{doc_name}_chunks.json"
    json_path.write_text(
        json.dumps(export_data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    size_kb = json_path.stat().st_size / 1024
    print(f"Exported to {json_path} ({size_kb:.0f} KB)")
    return json_path


# Export the comparison results
if sample_md_path and "comparison_results" in dir():
    export_chunks_to_json(
        comparison_results, OUTPUT_DIR, sample_md_path.stem
    )
else:
    print("No comparison results to export")

💾 Exported to /content/drive/MyDrive/RAG_Atlas200DK/data/output/ATCToolInstructions_chunks.json (673 KB)


In [ ]:
# ============================================================================
# Cell C.9 — Final Export: All Output Artifacts
# ============================================================================
export_artifact(OUTPUT_DIR, "chunking_output.zip")

print("\n" + "=" * 55)
print("✅ Pipeline complete!")
print("=" * 55)
print(f"\nArtifacts in {OUTPUT_DIR}:")
for f in sorted(OUTPUT_DIR.iterdir()):
    if f.is_file():
        print(f"  📄 {f.name} ({f.stat().st_size / 1024:.0f} KB)")